In [1]:
from pyspark.sql import SparkSession

spark = SparkSession. \
builder.\
appName("Bai tap thuc hanh Spark DF part 2"). \
config("spark.sql.warehouse.dir", "D:\Learn-spark\learn-spark-maide"). \
enableHiveSupport(). \
getOrCreate()

In [2]:
orders_df = spark.read.csv("D:/Learn-spark/learn-spark-maide/orders_wh.csv", header=True, inferSchema=True)

In [3]:
orders_df.show(5)

+--------+-------------------+-----------+---------------+
|order_id|         order_date|customer_id|   order_status|
+--------+-------------------+-----------+---------------+
|       1|2013-07-25 00:00:00|      11599|         CLOSED|
|       2|2013-07-25 00:00:00|        256|PENDING_PAYMENT|
|       3|2013-07-25 00:00:00|      12111|       COMPLETE|
|       4|2013-07-25 00:00:00|       8827|         CLOSED|
|       5|2013-07-25 00:00:00|      11318|       COMPLETE|
+--------+-------------------+-----------+---------------+
only showing top 5 rows



In [4]:
#Tao bang tam
orders_df.createOrReplaceTempView("orders_tmp")

In [5]:
spark.sql("select * from orders_tmp").show(5)

+--------+-------------------+-----------+---------------+
|order_id|         order_date|customer_id|   order_status|
+--------+-------------------+-----------+---------------+
|       1|2013-07-25 00:00:00|      11599|         CLOSED|
|       2|2013-07-25 00:00:00|        256|PENDING_PAYMENT|
|       3|2013-07-25 00:00:00|      12111|       COMPLETE|
|       4|2013-07-25 00:00:00|       8827|         CLOSED|
|       5|2013-07-25 00:00:00|      11318|       COMPLETE|
+--------+-------------------+-----------+---------------+
only showing top 5 rows



In [6]:
orders_df.groupBy("order_status").count().show()

+---------------+-----+
|   order_status|count|
+---------------+-----+
|PENDING_PAYMENT|15030|
|       COMPLETE|22899|
|        ON_HOLD| 3798|
| PAYMENT_REVIEW|  729|
|     PROCESSING| 8275|
|         CLOSED| 7556|
|SUSPECTED_FRAUD| 1558|
|        PENDING| 7610|
|       CANCELED| 1428|
+---------------+-----+



Viết code với data frame tương đối đơn giản, dễ hiểu

Viết bằng sprak sql

In [7]:
spark.sql("select order_status, count(*) from orders_tmp group by order_status").show()

+---------------+--------+
|   order_status|count(1)|
+---------------+--------+
|PENDING_PAYMENT|   15030|
|       COMPLETE|   22899|
|        ON_HOLD|    3798|
| PAYMENT_REVIEW|     729|
|     PROCESSING|    8275|
|         CLOSED|    7556|
|SUSPECTED_FRAUD|    1558|
|        PENDING|    7610|
|       CANCELED|    1428|
+---------------+--------+



2. Tim RA 10 KHÁCH HÀNG CÓ ORDER NHIỀU NHẤT
    

In [8]:
orders_df.groupBy("customer_id").count().sort("count", ascending = False).show(10)
#show() là phương thức để mình show ra và nhìn thôi


+-----------+-----+
|customer_id|count|
+-----------+-----+
|       5897|   16|
|      12431|   16|
|        569|   16|
|       6316|   16|
|      12284|   15|
|       5624|   15|
|        221|   15|
|       4320|   15|
|       5283|   15|
|       5654|   15|
+-----------+-----+
only showing top 10 rows



In [9]:
top_10_df = orders_df.groupBy("customer_id").count().sort("count", ascending = False).limit(10)
top_10_df.show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
|       5897|   16|
|      12431|   16|
|        569|   16|
|       6316|   16|
|      12284|   15|
|       5624|   15|
|       4320|   15|
|       5283|   15|
|        221|   15|
|       5654|   15|
+-----------+-----+



In [10]:
#Lam với spark sql
spark.sql("select customer_id, count(*) as count from orders_tmp group by customer_id order by count desc limit 10").show( )

+-----------+-----+
|customer_id|count|
+-----------+-----+
|       5897|   16|
|      12431|   16|
|        569|   16|
|       6316|   16|
|      12284|   15|
|       5624|   15|
|       4320|   15|
|       5283|   15|
|        221|   15|
|       5654|   15|
+-----------+-----+



3. ĐẾM SỐ LƯỢNG CUSTOMER LÀ BAO NHIÊU

In [11]:
#DF
orders_df.select("customer_id").distinct().count()

12405

In [12]:
#Lam voi spark sql
spark.sql("select count(distinct customer_id) as nub_customers from orders_tmp").show()

+-------------+
|nub_customers|
+-------------+
|        12405|
+-------------+



4. Tìm khách hàng có số lượng order CLOSE nhiều nhất

In [13]:
#DF
orders_df.where("order_status = 'CLOSED'").groupBy("customer_id").count().sort("count", ascending=False).show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
|       1833|    6|
|       1363|    5|
|       1687|    5|
|       5493|    5|
|       2236|    4|
|       3631|    4|
|       2774|    4|
|      10018|    4|
|      12431|    4|
|       7879|    4|
|        569|    4|
|       5319|    4|
|       9213|    4|
|       4573|    4|
|       4588|    4|
|      10263|    4|
|      10111|    4|
|       4997|    4|
|       7948|    4|
|       1443|    4|
+-----------+-----+
only showing top 20 rows



In [14]:
top_1 = orders_df.where("order_status = 'CLOSED'").groupBy("customer_id").count().sort("count", ascending=False).limit(1)
top_1.show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
|       1833|    6|
+-----------+-----+



In [15]:
#Lam voi spark sql
spark.sql("select customer_id, count(*) as count from orders_tmp where order_status = 'CLOSED' group by customer_id order by count desc limit 1").show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
|       1833|    6|
+-----------+-----+

